In [ ]:
%pip install -q transformer_lens "jlens @ git+https://github.com/anthropics/jacobian-lens.git"

import torch
from transformer_lens.model_bridge import TransformerBridge

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

dtype = torch.float32
print(f'Using {device=} and {dtype=}')

In [ ]:
model = TransformerBridge.boot_transformers(
    MODEL_NAME,
    device=device,
    dtype=dtype,
)
model.eval()
print(f'Loaded {MODEL_NAME}')

# Check that the model and TransformerLens cache work

In [ ]:
prompt = 'The capital of France is Paris'
tokens = model.to_tokens(prompt)

# Verify that TransformerLens hooks and the model both work.
with torch.no_grad():
    logits, _ = model.run_with_cache(tokens)

model.tokenizer.decode(logits[0,-1,:].argmax())


# read activations

In [ ]:
source_layer = model.cfg.n_layers // 2
target_layer = model.cfg.n_layers - 1

source_hook_name = f"blocks.{source_layer}.hook_resid_post"
target_hook_name = f"blocks.{target_layer}.hook_resid_post"

acts_dict = {}
def save_rs_fn(residual_stream, hook):
    acts_dict[hook.name] = residual_stream.detach()
    return residual_stream


In [ ]:
model.eval()
with torch.no_grad():
    logits = model.run_with_hooks(tokens, fwd_hooks=[ (source_hook_name, save_rs_fn), (target_hook_name, save_rs_fn)],)

In [ ]:
for k,v in acts_dict.items():
    print(f'source = {k}, shape = {v.shape}')

In [ ]:
penultimate_source_lyr = acts_dict['blocks.14.hook_out'][0, -2, :]
penultimate_target_lyr = acts_dict['blocks.27.hook_out'][0, -2, :]

print(f'penultimate_source_lyr = {penultimate_source_lyr.shape}')
print(f'penultimate_target_lyr = {penultimate_target_lyr.shape}')


# but since we need to calc jacobian, we need to store computation graph

In [ ]:
# we only need from source layer
for parameter in model.parameters():
    parameter.requires_grad_(False)


rs_dict_with_grad = {}
def source_hook_fn(rs, hook):
    detached_rs = rs.detach().requires_grad_(True)
    rs_dict_with_grad['source_full'] = detached_rs
    return detached_rs

def target_hook_fn(rs, hook):
    print(f'target={rs.shape}')
    rs_dict_with_grad['target_penultimate'] = rs[0, -2, :]
    return rs

In [ ]:
logits = model.run_with_hooks(tokens, fwd_hooks=[ (source_hook_name, source_hook_fn), (target_hook_name, target_hook_fn) ])
source_tensor = rs_dict_with_grad['source_full']
target_vector = rs_dict_with_grad['target_penultimate']

print(source_tensor.shape)
print(target_vector.shape)

jacobian_rows = []
for i in range(target_vector.numel()):
    grad_full = torch.autograd.grad(
        target_vector[i],
        source_tensor,
        retain_graph=True
    )
    penultimate_grad = grad_full[0][0, -2, :]
    jacobian_rows.append(penultimate_grad)
# Each row is d(target feature i) / d(source feature vector).

In [ ]:
jacobian_tensor = torch.stack(jacobian_rows, dim=0)
print(jacobian_tensor.shape)

In [ ]:
jacobian_tensor.shape

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(5,5))
J = jacobian_tensor.detach().float().cpu()
limit = torch.quantile(J.abs(), 0.99).item()
plt.imshow(jacobian_tensor.to(torch.float32).cpu().numpy(), vmin = -limit, vmax = limit, cmap='coolwarm')
plt.colorbar()
plt.title('jacobian calc through py torch')

In [ ]:
j_flat = jacobian_tensor.detach().float().cpu().flatten()
plt.hist(j_flat, bins=500);
plt.xlim(-0.2, 0.2);

# check with perturbation

In [ ]:
print(source_hook_name)
print(target_hook_name)

In [ ]:
plus_minus_eta_target = {}

eta = torch.randn(jacobian_tensor.shape[1], device=device, dtype=torch.float32)
eta = eta / eta.norm()

def perturb_source_plus_fn(rs, hook):
    patched = rs.clone()
    patched[0, -2, :] = patched[0, -2, :] + eta
    return patched

def perturb_source_minus_fn(rs, hook):
    patched = rs.clone()
    patched[0, -2, :] = patched[0, -2, :] - eta
    return patched

def perturb_target_plus_fn(rs, hook):
    plus_minus_eta_target['plus'] = rs[0, -2, :].detach().clone()
    return rs

def perturb_target_minus_fn(rs, hook):
    plus_minus_eta_target['minus'] = rs[0, -2, :].detach().clone()
    return rs


In [ ]:
with torch.no_grad():
    logits1 = model.run_with_hooks(
        tokens,
        fwd_hooks=[
            (source_hook_name, perturb_source_plus_fn),
            (target_hook_name, perturb_target_plus_fn),
        ],
    )

In [ ]:
with torch.no_grad():
    logits2 = model.run_with_hooks(
        tokens,
        fwd_hooks=[
            (source_hook_name, perturb_source_minus_fn),
            (target_hook_name, perturb_target_minus_fn),
        ],
    )

In [ ]:
plus_minus_eta_target.keys()

For a perturbation `eta`, the central finite difference predicts:

`0.5 * (f(x + eta) - f(x - eta))` should approximately equal `J @ eta`.

In [ ]:
# lhs
fx_plus_del_x = plus_minus_eta_target['plus']
fx_minus_del_x = plus_minus_eta_target['minus']

lhs = 0.5 * (fx_plus_del_x - fx_minus_del_x)

In [ ]:
rhs = jacobian_tensor @ eta

In [ ]:
print(lhs.shape)
print(rhs.shape)

In [ ]:
# check norms and angles
print(f'lhs norm = {lhs.norm()}, rhs norm = {rhs.norm()}')
# using bf16 format can cause a huge difference

In [ ]:
import torch.nn.functional as F
cosine_sim = F.cosine_similarity(lhs, rhs,dim=0)
print(f'cosine_sim = {cosine_sim.item()}')

# using anthropic's lib

In [ ]:
import jlens

`jlens.from_hf` expects the native Hugging Face model. `model.original_model` is the same Qwen model wrapped by `TransformerBridge`; it is not a copy.

In [ ]:

og_model = model.original_model

In [ ]:
jl_model = jlens.from_hf(
    og_model,
    model.tokenizer,
    force_bos = False, # we used same tokenization
)

In [ ]:
jl_model

In [ ]:
source_layer_id = model.cfg.n_layers // 2
target_layer_id = model.cfg.n_layers - 1

print(f'source = {source_layer_id}, target = {target_layer_id}')


In [ ]:
tokens[0].shape[0]

In [ ]:
from jlens.fitting import valid_position_mask

jlens_tokens = jl_model.encode(prompt)
assert torch.equal(tokens.cpu(), jlens_tokens.cpu()), 'Tokenizations differ'
len_tokens = jlens_tokens.shape[1]

position_mask = valid_position_mask(len_tokens, skip_first=len_tokens - 2)
print(position_mask)

`valid_position_mask` selects the sequence positions included in Anthropic's Jacobian estimator. Setting `skip_first=len_tokens-2` leaves only the penultimate position, so this run is directly comparable to the handwritten penultimate-to-penultimate Jacobian. The standard fitted lens instead averages over many valid positions and prompts.

In [ ]:
from jlens.fitting import jacobian_for_prompt

jlens_jacobian, jlib_seq_len, n_valid = jacobian_for_prompt(
    jl_model,
    prompt,
    source_layers = [source_layer_id],
    target_layer = target_layer_id,
    skip_first=len_tokens - 2,  # include only the penultimate position
)

In [ ]:
jlens_jacobian[source_layer_id].shape

In [ ]:
J1 = jacobian_tensor.detach().float().cpu()
J2 = jlens_jacobian[source_layer_id].detach().float().cpu()


In [ ]:
limit1 = torch.quantile(J1.abs(), 0.99).item()
plt.figure()
plt.imshow(J1, vmin = -limit1, vmax = limit1, cmap='coolwarm')
plt.colorbar()
plt.title('jacobian calc through py torch')

limit2 = torch.quantile(J2.abs(), 0.99).item()
plt.figure()
plt.imshow(J2, vmin = -limit2, vmax = limit2, cmap = 'coolwarm')
plt.colorbar()
plt.title('jacobian through jlens lib')

In [ ]:
# Use float64 reductions so rounding cannot push cosine slightly above 1.
j1_flat = J1.flatten().double()
j2_flat = J2.flatten().double()
difference = j1_flat - j2_flat

print(f'norm: handwritten={j1_flat.norm()}, jlens={j2_flat.norm()}')
print(f'norm ratio = {j2_flat.norm() / j1_flat.norm()}')
print(f'cosine = {F.cosine_similarity(j1_flat, j2_flat, dim=0)}')
print(f'relative L2 error = {difference.norm() / j1_flat.norm()}')
print(f'max absolute error = {difference.abs().max()}')

# Appendix

## bf16 vs f32

In [ ]:
a = torch.tensor(16.0, dtype=torch.bfloat16)
b = torch.tensor(16.0, dtype=torch.float32)

print(f'bf16 = {a + 0.03}') # 16.0
print(f'f32 = {b + 0.03}')  # approximately 16.03

every floating point format rounds off, but its about how much

| Format   | Sign | Exponent | Fraction | Total Bits |
|----------|-----:|---------:|---------:|-----------:|
| bfloat16 | 1    | 8        | 7        | 16         |
| float32  | 1    | 8        | 23       | 32         |


so bf16 gives less bits to fraction, so its rounds off more than float32, that's why one should use float32


  



## we calculate jacobian at penulimate token, not ultimate token

because when u pass "the capital of frace is paris", looking at the ultimate token means looking what comes after "paris" , which is not in our sample. Instead we should look at the penultimate token which is "is " which gives us info about the next token model is going to say.